# 回归分析

## 目标
通过多元线性回归与有序 Logit 回归识别分期意愿的驱动因素。

## 输入
- `data/processed/survey_cleaned.xlsx`

## 输出
- `results/analysis_output.xlsx`

## 对应方法
- OLS 多元线性回归
- Ordered Logit 稳健性检验

In [1]:
# 环境准备：读取数据并引入 src 回归模块
import sys
from pathlib import Path

import pandas as pd

sys.path.append("../src")
from regression import run_ols, run_ordered_logit

df = pd.read_excel(Path("../data/processed/survey_cleaned.xlsx"))
print("形状：", df.shape)

形状： (100, 30)


In [2]:
# 结果导出所需的统计量（在本 notebook 内重算，保证独立运行）
rank_cols = ["living_num","disposable_num","impulse_rank","demand_rank",
             "demand4_rank","trust_rank","cooling_rank",
             "installment_rank","shortage_rank","limit_rank"]
corr_matrix = df[rank_cols].corr(method="spearman")

scene_options = ["文娱票务","潮玩周边","户外与娱乐用品","假期旅游","数码电子","服饰美妆","二手交易"]

desc = df[["living_num","disposable_num"]].describe().T
desc["方差"] = df[["living_num","disposable_num"]].var().values
desc["标准差"] = df[["living_num","disposable_num"]].std().values
desc["偏度"] = df[["living_num","disposable_num"]].skew().values
desc["峰度"] = df[["living_num","disposable_num"]].kurt().values

In [3]:
# 多元线性回归：识别分期意愿的驱动因素
X = df[["living_num","disposable_num","impulse_rank","trust_rank","cooling_rank"]].copy()
y = df["demand4_rank"]

model = run_ols(df, "demand4_rank", list(X.columns))
print(model.summary())

print("\n===== 回归关键结果 =====")
print(f"R² = {model.rsquared:.4f}")
print(f"调整 R² = {model.rsquared_adj:.4f}")
print(f"F = {model.fvalue:.4f}, p = {model.f_pvalue:.6f}")
for var in model.params.index:
    if var == "const":
        continue
    coef, pval = model.params[var], model.pvalues[var]
    sig = "显著" if pval < 0.05 else "不显著"
    print(f"{var}：β={coef:.4f}, p={pval:.4f}, {sig}")

                            OLS Regression Results                            
Dep. Variable:           demand4_rank   R-squared:                       0.280
Model:                            OLS   Adj. R-squared:                  0.241
Method:                 Least Squares   F-statistic:                     7.303
Date:                Thu, 15 Oct 2026   Prob (F-statistic):           8.07e-06
Time:                        22:39:03   Log-Likelihood:                -97.719
No. Observations:                 100   AIC:                             207.4
Df Residuals:                      94   BIC:                             223.1
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.6128      0.452      1.

In [4]:
# 有序 Logit 回归：稳健性检验
X_logit = df[["living_num","disposable_num","impulse_rank","trust_rank","cooling_rank"]]
y_logit = df["demand4_rank"]

model_logit = run_ordered_logit(df, "demand4_rank", list(X_logit.columns))
print(model_logit.summary())

Optimization terminated successfully.
         Current function value: 0.849455
         Iterations: 38
         Function evaluations: 44
         Gradient evaluations: 44
                             OrderedModel Results                             
Dep. Variable:           demand4_rank   Log-Likelihood:                -84.945
Model:                   OrderedModel   AIC:                             183.9
Method:            Maximum Likelihood   BIC:                             202.1
Date:                Thu, 15 Oct 2026                                         
Time:                        22:39:03                                         
No. Observations:                 100                                         
Df Residuals:                      93                                         
Df Model:                           5                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------

In [5]:
# 分析结果导出
from scipy import stats

out_path = Path("../results/analysis_output.xlsx")
out_path.parent.mkdir(parents=True, exist_ok=True)

try:
    with pd.ExcelWriter(out_path) as writer:
        # 描述统计
        desc.to_excel(writer, sheet_name="描述统计")

        # 频数表
        for col in ["age","impulse","demand_4","cooling","focus","living",
                    "disposable","installment_use","shortage","limit_accept",
                    "demand","trust"]:
            df[col].value_counts().to_frame("频数").assign(
                比例=lambda x: (x["频数"]/len(df)*100).round(2)
            ).to_excel(writer, sheet_name=f"频数_{col}")

        # 多选题
        scene_df = pd.DataFrame({
            "选项": scene_options,
            "人数": [df[f"scene_{o}"].sum() for o in scene_options],
        })
        scene_df["普及率"] = (scene_df["人数"]/len(df)*100).round(2)
        scene_df.to_excel(writer, sheet_name="多选题_非计划开销", index=False)

        # Spearman 相关矩阵
        corr_matrix.round(3).to_excel(writer, sheet_name="Spearman相关矩阵")

        # 卡方结果
        chi_rows = []
        for a, b in [("living","demand_4"),("disposable","demand_4"),
                     ("impulse","cooling"),("installment_use","cooling"),
                     ("age","installment_use"),("trust","demand_4")]:
            table = pd.crosstab(df[a], df[b])
            chi2, p, dof, _ = stats.chi2_contingency(table)
            chi_rows.append({"变量1":a,"变量2":b,"卡方值":round(chi2,4),
                             "自由度":dof,"p值":round(p,4),
                             "结论":"显著相关" if p<0.05 else "不显著"})
        pd.DataFrame(chi_rows).to_excel(writer, sheet_name="卡方检验", index=False)

        # 回归结果
        reg_rows = []
        for var in X.columns:
            if var == "const":
                continue
            reg_rows.append({
                "变量": var,
                "系数β": round(model.params[var],4),
                "p值": round(model.pvalues[var],4),
                "显著性": "显著" if model.pvalues[var]<0.05 else "不显著"
            })
        reg_summary = pd.DataFrame(reg_rows)
        reg_summary.loc[len(reg_summary)] = ["R²", round(model.rsquared,4), "", ""]
        reg_summary.loc[len(reg_summary)] = ["调整R²", round(model.rsquared_adj,4), "", ""]
        reg_summary.loc[len(reg_summary)] = ["F值", round(model.fvalue,4), round(model.f_pvalue,6), ""]
        reg_summary.to_excel(writer, sheet_name="回归结果", index=False)

    print("已导出：", out_path)
except PermissionError:
    print("文件被占用，请关闭 Excel 后重试")

已导出： ..\results\analysis_output.xlsx
